In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets torch scikit-learn

In [ ]:
from pathlib import Path
import os
import random
import re

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def find_dataset_path() -> Path:
    drive_root = Path('/content/drive/MyDrive')
    candidates = list(drive_root.rglob('final_dataset.csv'))
    if not candidates:
        raise FileNotFoundError('final_dataset.csv was not found under /content/drive/MyDrive')
    return candidates[0]

def detect_columns(frame: pd.DataFrame) -> tuple[str, str]:
    code_candidates = ['code', 'snippet', 'text', 'source_code', 'content']
    label_candidates = ['label', 'type', 'authorship', 'class', 'target']
    code_col = next((column for column in code_candidates if column in frame.columns), None)
    label_col = next((column for column in label_candidates if column in frame.columns), None)
    if code_col is None or label_col is None:
        raise ValueError(f'Could not infer code and label columns from: {list(frame.columns)}')
    return code_col, label_col

def normalize_label(value) -> int:
    text = str(value).strip().lower()
    human_values = {'0', 'human', 'human-written', 'human_written', 'written', 'manual'}
    ai_values = {'1', 'ai', 'ai-generated', 'ai_generated', 'generated', 'machine', 'machine-generated', 'machine_generated'}
    if text in human_values:
        return 0
    if text in ai_values:
        return 1
    try:
        return int(float(text))
    except ValueError as error:
        raise ValueError(f'Unsupported label value: {value!r}') from error

DATA_PATH = find_dataset_path()
frame = pd.read_csv(DATA_PATH)
code_column, label_column = detect_columns(frame)
frame = frame[[code_column, label_column]].dropna().rename(columns={code_column: 'code', label_column: 'label'})
frame['code'] = frame['code'].astype(str)
frame['label'] = frame['label'].apply(normalize_label).astype(int)
train_df, test_df = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=frame['label'])
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df.head()

In [ ]:
TOKEN_PATTERN = re.compile(r'[A-Za-z_]\w*|\d+\.\d+|\d+|==|!=|<=|>=|->|::|<<|>>|//|/\*|\*/|[^\s]')
MAX_LENGTH = 512

def tokenize_code(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(str(text))

vocab = {'<pad>': 0, '<unk>': 1}
for code in train_df['code']:
    for token in tokenize_code(code):
        if token not in vocab:
            vocab[token] = len(vocab)

class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, data_frame: pd.DataFrame, vocabulary: dict[str, int], max_length: int = MAX_LENGTH):
        self.data_frame = data_frame.reset_index(drop=True)
        self.vocabulary = vocabulary
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.data_frame)

    def __getitem__(self, index: int):
        row = self.data_frame.iloc[index]
        token_ids = [self.vocabulary.get(token, self.vocabulary['<unk>']) for token in tokenize_code(row['code'])][: self.max_length]
        if len(token_ids) < self.max_length:
            token_ids.extend([self.vocabulary['<pad>']] * (self.max_length - len(token_ids)))
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(row['label'], dtype=torch.float32)

train_split_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df['label'])
train_dataset = CodeDataset(train_split_df.reset_index(drop=True), vocab)
val_dataset = CodeDataset(val_df.reset_index(drop=True), vocab)
test_dataset = CodeDataset(test_df, vocab)

train_dataset[0] if len(train_dataset) else None

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader

class TextCNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 128, num_filters: int = 100):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, kernel_size) for kernel_size in (3, 4, 5)])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * 3, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids).transpose(1, 2)
        features = [torch.relu(conv(embedded)) for conv in self.convs]
        pooled = [torch.max(feature, dim=2).values for feature in features]
        concatenated = torch.cat(pooled, dim=1)
        logits = self.fc(self.dropout(concatenated)).squeeze(1)
        return self.sigmoid(logits)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TextCNN(len(vocab)).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model

In [ ]:
def run_epoch(loader: DataLoader, training: bool = False):
    model.train(training)
    total_loss = 0.0
    predictions = []
    targets = []

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        batch_predictions = (outputs >= 0.5).long().detach().cpu().numpy().tolist()
        batch_targets = labels.long().detach().cpu().numpy().tolist()
        predictions.extend(batch_predictions)
        targets.extend(batch_targets)

    average_loss = total_loss / max(1, len(loader.dataset))
    accuracy = accuracy_score(targets, predictions)
    f1 = f1_score(targets, predictions)
    return average_loss, accuracy, f1

for epoch in range(1, 11):
    train_loss, train_accuracy, train_f1 = run_epoch(train_loader, training=True)
    val_loss, val_accuracy, val_f1 = run_epoch(val_loader, training=False)
    print(
        f'Epoch {epoch:02d} | '
        f'Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f} | Train F1: {train_f1:.4f} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f} | Val F1: {val_f1:.4f}'
    )

In [ ]:
model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        predictions = (outputs >= 0.5).long().cpu().numpy().tolist()
        targets = labels.long().numpy().tolist()
        all_predictions.extend(predictions)
        all_targets.extend(targets)

print('Confusion Matrix:')
print(confusion_matrix(all_targets, all_predictions))
print('
Classification Report:')
print(classification_report(all_targets, all_predictions, target_names=['Human-Written', 'AI-Generated']))

In [ ]:
output_dir = Path('/content/drive/MyDrive/models')
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / 'textcnn_weights.pt'
torch.save(
    {
        'state_dict': model.state_dict(),
        'vocab': vocab,
        'embed_dim': 128,
        'num_filters': 100,
        'max_length': MAX_LENGTH,
    },
    checkpoint_path,
)
print(f'Saved TextCNN checkpoint to {checkpoint_path}')